# 03 · Federated training with FedAvg
In a real federation the hospitals cannot pool raw patient records. **Federated learning** trains a shared model by exchanging *model weights*, not data. We use a transparent NumPy **FedAvg**: broadcast the global weights, each site takes a few local gradient steps, the server averages the results (weighted by site size).

In [ ]:
# --- workshop bootstrap: make the package importable ---
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join('..', 'src')))
import numpy as np, pandas as pd
from fedconformal import data, conformal, evaluate as ev, federated, heterogeneity as het, viz, eda
viz.set_style()
%matplotlib inline
# load the federation once (each notebook is self-contained)
df = data.load_raw()
sites = data.load_sites(shared_scaler=True)

## Hold a site out as an 'external' hospital
We train on Cleveland, Hungary and the V.A., and keep **Switzerland** completely unseen — the classic external-validation scenario a curation pipeline must survive.

In [ ]:
train_sites = ['cleveland', 'hungarian', 'va']
fed = federated.federated_averaging(sites, rounds=40, local_epochs=3,
                                    train_sites=train_sites, seed=0)
viz.plot_fed_learning_curves(fed.history);
global_model = fed.global_model

The per-site loss curves fall together as the shared model improves. Note the model never saw Switzerland, yet we will still ask it (and its conformal calibration) to behave there.

## Federated vs. centralized
As a reference, compare against a model that (hypothetically) pooled the same three sites.

In [ ]:
central = federated.train_centralized(sites, train_sites=train_sites, epochs=400)
def acc(m, s):
    p = m.predict_proba(sites[s].X)[:,1] > 0.5
    return float((p == sites[s].y).mean())
pd.DataFrame({
    'federated': {s: round(acc(global_model, s),3) for s in data.SITES},
    'centralized': {s: round(acc(central, s),3) for s in data.SITES},
})

Accuracy is similar — FedAvg recovers most of the centralized performance **without moving data**. But accuracy alone hides *where the model is unreliable*. That is the job of conformal prediction, in the next notebook.

### Exercise
Add Switzerland to `train_sites` and re-run. Does including the most-shifted site help or hurt the other sites? Relate your answer to the domain-AUC matrix from notebook 01.